In [0]:
# =============================================================================
# jobs/05_refresh_boundaries.py
# Quarterly refresh of parliamentary constituency boundaries.
#
# What this script does:
#   1. Fetches current constituency boundaries from the ONS Open Geography Portal.
#   2. Stores each constituency as a row with a GeoJSON geometry string.
#   3. Overwrites parliamentary_constituencies in full.
#
# Run schedule: quarterly, or immediately after a general election.
# After this job completes, run 06_compute_lookup.py to recompute the
# EA area / constituency spatial join.
#
# GeoPandas reads the GeoJSON directly from the ONS URL in a single call.
# No Mosaic or spatial library beyond GeoPandas and Shapely is needed.
# =============================================================================

import sys
import json

sys.path.insert(0, "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/")
from config import ONS_CONSTITUENCIES_URL, TBL_CONSTITUENCIES
from utils.helpers import get_spark, utc_now

import geopandas as gpd
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()


# =============================================================================
# SCHEMA
# Defined explicitly to avoid type inference issues with None values.
# =============================================================================

constituency_schema = StructType([
    StructField("constituency_id",   StringType(), True),
    StructField("name",              StringType(), True),
    StructField("geometry",          StringType(), True),
    StructField("last_refreshed_at", StringType(), True),
])




In [0]:
# =============================================================================
# FETCH BOUNDARIES
# GeoPandas reads the GeoJSON FeatureCollection directly from the ONS URL.
# The geometry column is populated automatically from each feature's geometry.
#
# The ONS URL in config.py requests generalised (BGC) boundaries.
# These are appropriate for spatial joins and display -- not surveying.
# If you need full precision boundaries, update the URL in config.py to BFE.
# =============================================================================

print(f"Fetching constituency boundaries from ONS...")
print(f"URL: {ONS_CONSTITUENCIES_URL}")

gdf = gpd.read_file(ONS_CONSTITUENCIES_URL)
print(f"Retrieved {len(gdf)} constituency features.")
print(f"Columns: {gdf.columns.tolist()}")




In [0]:
# =============================================================================
# PARSE FEATURES
# The ONS GeoJSON properties include the GSS code and name.
# Field names include the year of the boundary release (24 = 2024).
# If you update to a newer boundary release, check field names have not changed.
# =============================================================================

# Detect the correct column names -- they vary by release year.
# Try 2024 names first, then fall back to older patterns.
id_col   = next((c for c in gdf.columns if c.startswith("PCON") and c.endswith("CD")), None)
name_col = next((c for c in gdf.columns if c.startswith("PCON") and c.endswith("NM")), None)

if not id_col or not name_col:
    raise Exception(
        f"Could not find constituency ID or name columns. "
        f"Available columns: {gdf.columns.tolist()}"
    )

print(f"Using ID column: {id_col}, name column: {name_col}")

constituency_rows = []

for _, feature in gdf.iterrows():
    # Serialise the Shapely geometry to a GeoJSON string.
    # __geo_interface__ returns a dict; json.dumps converts it to a string.
    geometry_str = json.dumps(feature.geometry.__geo_interface__) \
        if feature.geometry else None

    constituency_rows.append(Row(
        constituency_id   = feature.get(id_col),
        name              = feature.get(name_col),
        geometry          = geometry_str,
        last_refreshed_at = now_iso
    ))

print(f"Parsed {len(constituency_rows)} rows.")




In [0]:
# =============================================================================
# WRITE TO DELTA
# =============================================================================

if not constituency_rows:
    raise Exception("No constituency rows parsed -- check ONS URL and column names.")

const_df = spark.createDataFrame(constituency_rows, schema=constituency_schema)
print(f"DataFrame has {const_df.count()} rows.")

(
    const_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_CONSTITUENCIES)
)

written = spark.table(TBL_CONSTITUENCIES).count()
print(f"Verified: {written} rows written to {TBL_CONSTITUENCIES}.")

print("Constituency boundary refresh complete.")
dbutils.notebook.exit("success")
